# QCoDeS Example with QuTech M2m Preamplifier

This notebook explains how the QuTech M2m voltage preamplifier module works and shows the main features of its QCoDeS driver.

## QuTech M2m Voltage Preamplifier

The M2m is a low-noise voltage amplifier designed for use in the IVVI rack system. It amplifies small voltage signals with selectable gain.

**Key Features:**
- Voltage gains: 1, 10, 100, 1k, 10k (V/V)
- DC or AC coupling mode
- Compatible with IVVI rack system

**Documentation:** https://qtwork.tudelft.nl/~schouten/ivvi/doc-mod/docm2m.htm

## Virtual Driver

This is a **virtual driver** - it does not communicate with the physical instrument. It is the user's responsibility to manually set the physical instrument to match the driver settings. The driver helps track settings and calculate the total gain.

## Quick Start: Measurement Example

### Import Required Libraries

In [1]:
from qcodes_contrib_drivers.drivers.QuTech.M2m import M2m, VoltageParameter
import numpy as np

### Create an M2m Instance

In [2]:
preamp = M2m("m2m_preamp")

### Setting Module Slot

The M2m module can be placed in slot Ma (top, iso-out 1) or Mb (bottom, iso-out 2):

In [3]:
preamp.slot("Ma")

### Setting Voltage Gain

The main gain setting amplifies the input voltage. Choose from: "1", "10", "100", "1k", "10k"

In [4]:
# Set gain to 1000 (1k V/V)
preamp.gain("1k")

### Setting DC/AC Coupling Mode

The M2m can operate in DC or AC coupling mode. Options: "dc", "ac", "hpf" (high-pass filter, if available)

In [5]:
# Set to DC coupling for DC measurements
preamp.dc_ac_mode("dc")

### Using VoltageParameter for Measurements

The `VoltageParameter` class automatically converts amplified voltage measurements back to the input voltage using the M2m's total gain.

In [6]:
from qcodes.instrument_drivers.mock_instruments import DummyInstrument

# Create a mock DMM (digital multimeter) that simulates measuring the M2m output
# In a real setup, this would be an actual DMM like Keysight 34465A
dmm = DummyInstrument(name="dmm", gates=["voltage"])

# Create VoltageParameter that links voltage measurement to M2m
voltage_param = VoltageParameter(
    measured_param=dmm.voltage,  # Use the DMM voltage parameter
    voltage_amplifier_instrument=preamp,
    name="voltage"
)

In [7]:
# Create a mock signal source
signal_source = DummyInstrument(name="signal_source", gates=["voltage"])

# Simulate a voltage sweep measurement
print("Voltage sweep measurement:")
print("\nInput Voltage (mV) | M2m Output (V) | Measured Input (mV)")
print("-" * 60)

for v_input in np.arange(0.1, 1.0, 0.1):  # Input voltages in mV
    signal_source.voltage(v_input * 1e-3)  # Set input voltage
    
    # Simulate M2m output voltage (input * gain)
    # In reality, this would be measured by your DMM
    dmm.voltage(v_input * 1e-3 * preamp.total_gain())  # Amplified by current gain setting
    
    # Get voltage measurement (automatically calculated from amplified voltage)
    voltage_raw, voltage = voltage_param.get()
    
    print(f"{v_input:+18.1f} | {voltage_raw:14.3f} | {voltage*1e3:20.1f}")

print("\nMeasurement complete!")

Voltage sweep measurement:

Input Voltage (mV) | M2m Output (V) | Measured Input (mV)
------------------------------------------------------------
              +0.1 |          0.100 |                  0.1
              +0.2 |          0.200 |                  0.2
              +0.3 |          0.300 |                  0.3
              +0.4 |          0.400 |                  0.4
              +0.5 |          0.500 |                  0.5
              +0.6 |          0.600 |                  0.6
              +0.7 |          0.700 |                  0.7
              +0.8 |          0.800 |                  0.8
              +0.9 |          0.900 |                  0.9

Measurement complete!


## Cleanup

In [8]:
# Clean up mock instruments
signal_source.close()
dmm.close()
# Close the instrument when done
preamp.close()